# Disaster Tweet Classification: Word2Vec Deep Recurrent Benchmark

**Embeddings:** Domain-Trained Word2Vec (Skip-Gram with Negative Sampling, 300-dim)  
**Architectures:** Simple RNN, BiRNN, 2-Stacked BiRNN, LSTM, BiLSTM, 2-Stacked BiLSTM  
**Objective:** End-to-end multi-architecture recurrent benchmark, per-model artifact serialization, and comparison across 10 humanitarian crisis categories.

---
### Notebook Structure
1. **Dataset Ingestion & Preprocessing**
2. **Domain-Specific Word2Vec Embedding Training & Alignment**
3. **PyTorch Dataset Pipeline & Token Indexing**
4. **Modular Recurrent Neural Network Architectures**
5. **Per-Model Artifact Serialization (`models/<model_name>/` and `saved_models/`)**
6. **Comprehensive Metrics Comparison (`metrics_comparison.csv` and `models_metrics_comparison.png`)**

In [ ]:
import os
import sys
import json
import random
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from gensim.models import Word2Vec
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Using compute device: {device}")

# Setup directories
DATA_DIR = Path("dataset")
RESULTS_DIR = Path("results/02_word2vec_recurrent_models")
MODELS_DIR = RESULTS_DIR / "saved_models"
PER_MODEL_DIR = RESULTS_DIR / "models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PER_MODEL_DIR, exist_ok=True)

train_path = DATA_DIR / "train_clean.parquet"
val_path = DATA_DIR / "validation_clean.parquet"
test_path = DATA_DIR / "test_clean.parquet"

# Sourcing dataset
KAGGLE_INPUT_DIR = Path("/kaggle/input/humaid-disaster-tweets-parquet")
if KAGGLE_INPUT_DIR.exists():
    print("[+] Sourcing dataset from Kaggle dataset input...")
    for split in ["train", "validation", "test"]:
        p_clean = KAGGLE_INPUT_DIR / f"{split}_clean.parquet"
        p_raw = KAGGLE_INPUT_DIR / f"{split}.parquet"
        target_p = DATA_DIR / f"{split}_clean.parquet"
        if not target_p.exists():
            if p_clean.exists():
                pd.read_parquet(p_clean).to_parquet(target_p)
            elif p_raw.exists():
                pd.read_parquet(p_raw).to_parquet(target_p)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    raw_train = DATA_DIR / "train.parquet"
    raw_val = DATA_DIR / "validation.parquet"
    raw_test = DATA_DIR / "test.parquet"
    if not (raw_train.exists() and raw_val.exists() and raw_test.exists()):
        print("[+] Downloading HumAID dataset from Google Drive...")
        import gdown
        GDRIVE_URL = "https://drive.google.com/drive/folders/1pyMBc4SFc-sQvfmReiywPoN5cQbMpQBR?usp=drive_link"
        gdown.download_folder(url=GDRIVE_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)

train_file = train_path if train_path.exists() else DATA_DIR / "train.parquet"
val_file = val_path if val_path.exists() else DATA_DIR / "validation.parquet"
test_file = test_path if test_path.exists() else DATA_DIR / "test.parquet"

train_df = pd.read_parquet(train_file)
val_df = pd.read_parquet(val_file)
test_df = pd.read_parquet(test_file)

text_col = "clean_text" if "clean_text" in train_df.columns else "tweet_text"
print(f"[+] Loaded splits using column '{text_col}':")
print(f"    Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 1. Domain-Specific Word2Vec Embedding Training
We train a 300-dimensional Word2Vec model on the cleaned training tweets using Skip-Gram architecture with negative sampling.

In [ ]:
# Tokenize texts for Word2Vec training
def tokenize_tweet(text):
    if not isinstance(text, str):
        return []
    return text.lower().split()

train_sentences = [tokenize_tweet(t) for t in train_df[text_col]]
val_sentences = [tokenize_tweet(t) for t in val_df[text_col]]
test_sentences = [tokenize_tweet(t) for t in test_df[text_col]]

EMB_DIM = 300
print(f"[+] Training Word2Vec ({EMB_DIM}-dim, Skip-Gram, min_count=2)...")
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=EMB_DIM,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    epochs=15,
    seed=42
)

w2v_save_path = MODELS_DIR / "word2vec_300d.model"
w2v_model.save(str(w2v_save_path))
print(f"[+] Saved Word2Vec model to: {w2v_save_path}")
print(f"    Vocabulary Size: {len(w2v_model.wv.key_to_index):,} tokens")

# Build Vocab & Embedding Matrix for PyTorch
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word in w2v_model.wv.key_to_index:
    vocab[word] = len(vocab)

embedding_matrix = np.zeros((len(vocab), EMB_DIM), dtype=np.float32)
# Initialize UNK with normal distribution
embedding_matrix[1] = np.random.normal(scale=0.6, size=(EMB_DIM,))
for word, idx in vocab.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print(f"[+] Embedding Matrix Shape: {embedding_matrix.shape}")

## 2. PyTorch Dataset & DataLoaders

In [ ]:
MAX_LEN = 64
BATCH_SIZE = 64

class TweetRecurrentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=64):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
        self.unk_idx = vocab.get(UNK_TOKEN, 1)
        self.pad_idx = vocab.get(PAD_TOKEN, 0)
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        tokens = tokenize_tweet(self.texts[idx])
        indices = [self.vocab.get(t, self.unk_idx) for t in tokens][:self.max_len]
        if len(indices) < self.max_len:
            indices += [self.pad_idx] * (self.max_len - len(indices))
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

class_names = sorted(train_df["class_label"].unique())
label2idx = {name: i for i, name in enumerate(class_names)}
idx2label = {i: name for i, name in enumerate(class_names)}

y_train = train_df["class_label"].map(label2idx).values
y_val = val_df["class_label"].map(label2idx).values
y_test = test_df["class_label"].map(label2idx).values

train_dataset = TweetRecurrentDataset(train_df[text_col].values, y_train, vocab, MAX_LEN)
val_dataset = TweetRecurrentDataset(val_df[text_col].values, y_val, vocab, MAX_LEN)
test_dataset = TweetRecurrentDataset(test_df[text_col].values, y_test, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Class Imbalance Weights for Loss Function
class_counts = np.bincount(y_train, minlength=len(class_names))
class_weights = len(y_train) / (len(class_names) * class_counts.astype(np.float32))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print(f"[+] Loaded DataLoaders. Applied balanced CrossEntropyLoss weights.")

## 3. Modular Recurrent Neural Network Architecture

In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, embedding_matrix, cell_type="bilstm", hidden_dim=128, num_layers=1, num_classes=10, dropout=0.3):
        super().__init__()
        vocab_size, emb_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float), freeze=False)
        self.cell_type = cell_type.lower()
        self.bidirectional = "bi" in self.cell_type
        
        rnn_dropout = dropout if num_layers > 1 else 0.0
        
        if "lstm" in self.cell_type:
            self.rnn = nn.LSTM(
                emb_dim, hidden_dim, num_layers=num_layers,
                bidirectional=self.bidirectional, batch_first=True, dropout=rnn_dropout
            )
        else: # rnn
            self.rnn = nn.RNN(
                emb_dim, hidden_dim, num_layers=num_layers,
                bidirectional=self.bidirectional, batch_first=True, dropout=rnn_dropout, nonlinearity='tanh'
            )
            
        fc_in = hidden_dim * 2 if self.bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(fc_in, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        if "lstm" in self.cell_type:
            out, (hn, cn) = self.rnn(embedded)
        else:
            out, hn = self.rnn(embedded)
            
        pooled, _ = torch.max(out, dim=1)
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)
        return logits

print("[+] Defined RecurrentClassifier modular architecture.")

## 4. Training Engine & Per-Model Evaluation Helper

In [ ]:
display_name_map = {
    'caution_and_advice': 'Caution & Advice',
    'displaced_people_and_evacuations': 'Displaced / Evac.',
    'infrastructure_and_utility_damage': 'Infrastructure',
    'injured_or_dead_people': 'Injured / Dead',
    'missing_or_found_people': 'Missing / Found',
    'not_humanitarian': 'Not Humanitarian',
    'other_relevant_information': 'Other Info',
    'requests_or_urgent_needs': 'Requests / Urgent',
    'rescue_volunteering_or_donation_effort': 'Rescue / Donation',
    'sympathy_and_support': 'Sympathy / Support'
}
display_names = [display_name_map.get(c, c) for c in class_names]

# Helper function to save complete per-model artifacts
def save_model_evaluation_artifacts(model_name, y_true, y_pred, history, metrics_dict):
    model_slug = model_name.lower().replace(" ", "_").replace("+", "plus").replace("-", "_")
    curr_model_dir = PER_MODEL_DIR / model_slug
    os.makedirs(curr_model_dir, exist_ok=True)
    
    # 1. Classification Report
    report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    with open(curr_model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(f"=== Classification Report: {model_name} ===\n\n")
        f.write(report_str)
        
    # 2. Metrics JSON
    with open(curr_model_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_dict, f, indent=2)
        
    # 3. Dual Confusion Matrix Plot (Clean formatted labels)
    cm_raw = confusion_matrix(y_true, y_pred)
    cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]
    
    display_names = [
        'Caution & Advice', 'Displaced / Evac.', 'Infrastructure',
        'Injured / Dead', 'Missing / Found', 'Not Humanitarian',
        'Other Relevant', 'Urgent Needs', 'Rescue / Donation', 'Sympathy & Support'
    ]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8.5), dpi=300)
    sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names,
                cbar=True, ax=ax1, annot_kws={"size": 8.5})
    ax1.set_title(f"Raw Confusion Matrix: {model_name}", fontsize=12, fontweight='bold', pad=12)
    ax1.set_xlabel("Predicted Class", fontsize=11, fontweight='bold', labelpad=8)
    ax1.set_ylabel("True Class", fontsize=11, fontweight='bold', labelpad=8)
    ax1.set_xticklabels(display_names, rotation=35, ha='right', fontsize=9.5)
    ax1.set_yticklabels(display_names, rotation=0, fontsize=9.5)
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names,
                cbar=True, ax=ax2, annot_kws={"size": 8.5})
    ax2.set_title(f"Normalized Confusion Matrix: {model_name}", fontsize=12, fontweight='bold', pad=12)
    ax2.set_xlabel("Predicted Class", fontsize=11, fontweight='bold', labelpad=8)
    ax2.set_ylabel("True Class", fontsize=11, fontweight='bold', labelpad=8)
    ax2.set_xticklabels(display_names, rotation=35, ha='right', fontsize=9.5)
    ax2.set_yticklabels(display_names, rotation=0, fontsize=9.5)
    
    plt.tight_layout()
    plt.savefig(curr_model_dir / "confusion_matrix.png", bbox_inches='tight')
    plt.close()
    
    # 4. Per-Class Precision, Recall, and F1 Bar Chart
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    per_class_df = pd.DataFrame([
        {
            "Class": cls,
            "Precision": report_dict[cls]["precision"],
            "Recall": report_dict[cls]["recall"],
            "F1-Score": report_dict[cls]["f1-score"],
            "Support": report_dict[cls]["support"]
        }
        for cls in class_names
    ])
    per_class_df.to_csv(curr_model_dir / "per_class_metrics.csv", index=False)
    
    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
    x = np.arange(len(class_names))
    width = 0.25
    ax.barh(x - width, per_class_df["Precision"], width, label="Precision", color="#3498db")
    ax.barh(x, per_class_df["Recall"], width, label="Recall", color="#2ecc71")
    ax.barh(x + width, per_class_df["F1-Score"], width, label="F1-Score", color="#e74c3c")
    ax.set_yticks(x)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel("Score", fontsize=10, fontweight='bold')
    ax.set_title(f"Per-Class Performance: {model_name}", fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(curr_model_dir / "per_class_metrics.png", bbox_inches='tight')
    plt.close()
    
    # 5. Training Curves Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
    epochs_range = range(1, len(history["train_loss"]) + 1)
    ax1.plot(epochs_range, history["train_loss"], 'b-o', label='Train Loss')
    ax1.plot(epochs_range, history["val_loss"], 'r-o', label='Val Loss')
    ax1.set_title(f"Loss Progression: {model_name}", fontsize=11, fontweight='bold')
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    
    ax2.plot(epochs_range, history["val_macro_f1"], 'g-o', label='Val Macro F1')
    ax2.set_title(f"Val Macro F1 Progression: {model_name}", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Macro F1")
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(curr_model_dir / "training_curves.png", bbox_inches='tight')
    plt.close()

def train_and_evaluate_model(model_name, cell_type, num_layers, epochs=10, lr=1e-3):
    print(f"\n=======================================================")
    print(f"[+] Training Model: {model_name} (cell={cell_type}, layers={num_layers})")
    print(f"=======================================================")
    
    model = RecurrentClassifier(
        embedding_matrix=embedding_matrix,
        cell_type=cell_type,
        hidden_dim=128,
        num_layers=num_layers,
        num_classes=len(class_names),
        dropout=0.3
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    
    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
    best_val_f1 = 0.0
    model_slug = model_name.lower().replace(' ', '_').replace('+', 'plus').replace('-', '_')
    best_weights_path = MODELS_DIR / f"{model_slug}.pt"
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
            
        train_loss = total_loss / len(train_dataset)
        
        # Validation Phase
        model.eval()
        val_loss, all_preds, all_labels = 0.0, [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss += loss.item() * len(y_batch)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y_batch.cpu().numpy())
                
        val_loss /= len(val_dataset)
        val_f1 = float(f1_score(all_labels, all_preds, average='macro', zero_division=0))
        scheduler.step(val_f1)
        
        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(float(val_loss))
        history["val_macro_f1"].append(float(val_f1))
        
        print(f"    Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_weights_path)
            
    # Load Best Model for Final Test Evaluation
    model.load_state_dict(torch.load(best_weights_path))
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_labels.extend(y_batch.numpy())
            
    test_acc = float(accuracy_score(test_labels, test_preds))
    test_f1_macro = float(f1_score(test_labels, test_preds, average='macro', zero_division=0))
    test_f1_weighted = float(f1_score(test_labels, test_preds, average='weighted', zero_division=0))
    test_prec_macro = float(precision_score(test_labels, test_preds, average='macro', zero_division=0))
    test_rec_macro = float(recall_score(test_labels, test_preds, average='macro', zero_division=0))
    
    metrics_dict = {
        "Model": model_name,
        "Cell Type": cell_type.upper(),
        "Layers": num_layers,
        "Val Best Macro F1": float(best_val_f1),
        "Test Accuracy": test_acc,
        "Test Macro F1": test_f1_macro,
        "Test Weighted F1": test_f1_weighted,
        "Test Macro Precision": test_prec_macro,
        "Test Macro Recall": test_rec_macro,
    }
    
    # Save complete per-model artifacts
    save_model_evaluation_artifacts(model_name, test_labels, test_preds, history, metrics_dict)
    
    print(f"[+] Final Test Results for {model_name}:")
    print(f"    --> Macro F1: {test_f1_macro:.4f} | Accuracy: {test_acc:.4f} | Weighted F1: {test_f1_weighted:.4f}")
    
    return metrics_dict

## 5. Run Benchmark across 6 Recurrent Architectures

In [ ]:
experiments = [
    {"name": "Word2Vec + Simple RNN", "cell_type": "rnn", "num_layers": 1, "epochs": 10},
    {"name": "Word2Vec + BiRNN", "cell_type": "birnn", "num_layers": 1, "epochs": 10},
    {"name": "Word2Vec + 2-Stacked BiRNN", "cell_type": "birnn", "num_layers": 2, "epochs": 10},
    {"name": "Word2Vec + LSTM", "cell_type": "lstm", "num_layers": 1, "epochs": 10},
    {"name": "Word2Vec + BiLSTM", "cell_type": "bilstm", "num_layers": 1, "epochs": 10},
    {"name": "Word2Vec + 2-Stacked BiLSTM", "cell_type": "bilstm", "num_layers": 2, "epochs": 10},
]

summary_metrics = []

for exp in experiments:
    res = train_and_evaluate_model(
        model_name=exp["name"],
        cell_type=exp["cell_type"],
        num_layers=exp["num_layers"],
        epochs=exp["epochs"]
    )
    summary_metrics.append(res)

summary_df = pd.DataFrame(summary_metrics).sort_values(by="Test Macro F1", ascending=False)
summary_df.to_csv(RESULTS_DIR / "metrics_comparison.csv", index=False)
print("\n=== Word2Vec Recurrent Architectures Benchmark Summary ===")
print(summary_df.to_string(index=False))

# Plot Comparison Figure of All Models in this Notebook (Test Macro F1 vs Accuracy)
df_sorted = summary_df.sort_values(by="Test Macro F1", ascending=False).reset_index(drop=True)

model_labels = []
for m in df_sorted["Model"]:
    m_label = m.replace("LogisticRegression", "LR").replace("MultinomialNB", "MNB").replace("ComplementNB", "CNB")
    model_labels.append(m_label)

x = np.arange(len(df_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(df_sorted) * 1.3), 6), dpi=300)
ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=1)

bars1 = ax.bar(x - width/2, df_sorted["Test Macro F1"], width, label="Macro F1", color="#1f77b4", edgecolor="none", zorder=3)
bars2 = ax.bar(x + width/2, df_sorted["Test Accuracy"], width, label="Accuracy", color="#ff7f0e", edgecolor="none", zorder=3)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f"{height:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8.5)
                
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f"{height:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8.5)

ax.set_title("Word2Vec Recurrent Benchmark: Test Macro F1 vs Accuracy", fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel("Model", fontsize=11, fontweight='bold', labelpad=10)
ax.set_ylabel("Score", fontsize=11, fontweight='bold', labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=35, ha='right', fontsize=9.5)

max_val = max(df_sorted["Test Macro F1"].max(), df_sorted["Test Accuracy"].max())
ax.set_ylim(0, min(1.0, max_val + 0.12))
ax.legend(loc="upper right", frameon=True, fontsize=9.5)

for spine in ax.spines.values():
    spine.set_color('#888888')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "models_metrics_comparison.png", bbox_inches='tight')
plt.show()

print(f"[+] All artifacts, models, reports, and comparison plots saved to: {RESULTS_DIR.resolve()}")